In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import torch
import traceback
import numpy as np
from runner import *
torch.set_printoptions(profile='full')

In [3]:
from easydict import EasyDict as edict
target_config_path = 'config/xor_neuron_mlp_cifar_test.yaml'

try:
    from runner import *
    from utils.logger import setup_logging
    from utils.arg_helper import get_config
except ImportError as e:
    raise e

args = edict()
args.config_file = target_config_path
args.test = False
args.log_level = 'INFO'
args.comment = 'notebook_run'


sample_id=1
config = get_config(args.config_file, sample_id="{:03d}".format(sample_id))

np.random.seed(config.seed)
torch.manual_seed(config.seed)
torch.cuda.manual_seed_all(config.seed)
config.use_gpu = config.use_gpu and torch.cuda.is_available()


log_file = os.path.join(config.save_dir, "log_exp_{}.txt".format(config.run_id))


logger = setup_logging(args.log_level, log_file)

logger.info("Writing log file to {}".format(log_file))
logger.info("Exp instance id = {}".format(config.run_id))
logger.info("Exp comment = {}".format(args.comment))

print(f"🚀 正在启动 Runner: {config.runner}")
print(f"📂 结果将保存到: {config.save_dir}")


runner = eval(config.runner)(config)


INFO  | 2025-12-24 22:12:53,678 | 3325531581.py             | line 32   : Writing log file to /home/yc/repo/xor_neuron_yoon/exp/xor_neuron_mlp_cifar_test/XorNeuronMLP_001_cifar10_2025-Dec-24-22-12-53/log_exp_922545.txt
INFO  | 2025-12-24 22:12:53,679 | 3325531581.py             | line 33   : Exp instance id = 922545
INFO  | 2025-12-24 22:12:53,680 | 3325531581.py             | line 34   : Exp comment = notebook_run


🚀 正在启动 Runner: XorNeuronRunner_test
📂 结果将保存到: /home/yc/repo/xor_neuron_yoon/exp/xor_neuron_mlp_cifar_test/XorNeuronMLP_001_cifar10_2025-Dec-24-22-12-53


In [4]:

if not args.test:
    # --- [Step 1] Pretrain ---
    print("\n--- [Step 1] Pretraining (InnerNet) ---")
    
    num_cells = config.model.num_cell_types
    print(f"💡 检测到配置需要预训练 {num_cells} 种细胞类型...")

    for i in range(num_cells):
        print(f"   > 正在预训练细胞类型: {i}")
        runner.pretrain(i)

    # --- [Step 2] Phase 1 ---
    # print("\n--- [Step 2] Training Phase 1 (Joint Training) ---")
    runner.train_phase1_v2()

    # # --- [Step 3] Phase 2 ---
    # print("\n--- [Step 3] Training Phase 2 (Freeze Activation) ---")
    runner.train_phase2()

    # # --- [Step 4] Testing ---
    runner.test_local()





--- [Step 1] Pretraining (InnerNet) ---
💡 检测到配置需要预训练 1 种细胞类型...
   > 正在预训练细胞类型: 0
Pretrain Loss @ epoch 0001 = 0.0076749485451728106
Pretrain Loss @ epoch 0002 = 0.0023654307413380595
Pretrain Loss @ epoch 0003 = 0.0017152648244518787
Pretrain Loss @ epoch 0004 = 0.0015591679897625
Pretrain Loss @ epoch 0005 = 0.0015022926847450436
Pretrain Loss @ epoch 0006 = 0.0014646539988461883
Pretrain Loss @ epoch 0007 = 0.0014508171414490789
Pretrain Loss @ epoch 0008 = 0.001433220860781148
Pretrain Loss @ epoch 0009 = 0.00141122923232615
Pretrain Loss @ epoch 0010 = 0.001379238412482664
Pretrain Loss @ epoch 0013 = 0.0013435650616884232
Pretrain Loss @ epoch 0014 = 0.001335158897563815
Pretrain Loss @ epoch 0015 = 0.001311125501524657
Pretrain Loss @ epoch 0016 = 0.0012891735008452089
Pretrain Loss @ epoch 0017 = 0.0012820011819712819
Pretrain Loss @ epoch 0018 = 0.0012807366787455977
Pretrain Loss @ epoch 0019 = 0.0012503745208960026
Pretrain Loss @ epoch 0020 = 0.0012463727442082017
Pretrain

/home/yc/repo/xor_neuron_yoon/utils/train_helper.py:36: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_snapshot = torch.load(file_name, map_location=device)
INFO  | 202

Files already downloaded and verified
Loaded model from /home/yc/repo/xor_neuron_yoon/exp/xor_neuron_mlp_cifar_test/XorNeuronMLP_001_cifar10_2025-Dec-24-22-12-53/model_snapshot_best_phase1.pth to cuda


100%|██████████| 20/20 [00:00<00:00, 39.47it/s]
INFO  | 2025-12-24 22:14:34,042 | inference_runner.py       | line 1110 : Avg. Validation Loss = 2.4001648306846617 +- 0
INFO  | 2025-12-24 22:14:34,045 | inference_runner.py       | line 1125 : Current Best Validation Loss = 2.4001648306846617
INFO  | 2025-12-24 22:14:36,465 | inference_runner.py       | line 1176 : Best Validation Loss = 2.4001648306846617
100%|██████████| 20/20 [00:00<00:00, 36.30it/s]
INFO  | 2025-12-24 22:14:37,019 | inference_runner.py       | line 1110 : Avg. Validation Loss = 1.9651971340179444 +- 0
INFO  | 2025-12-24 22:14:37,027 | inference_runner.py       | line 1125 : Current Best Validation Loss = 1.9651971340179444
INFO  | 2025-12-24 22:14:39,704 | inference_runner.py       | line 1176 : Best Validation Loss = 1.9651971340179444
100%|██████████| 20/20 [00:00<00:00, 36.19it/s]
INFO  | 2025-12-24 22:14:40,260 | inference_runner.py       | line 1110 : Avg. Validation Loss = 1.797316700220108 +- 0
INFO  | 2025-1

InnerNetData
train
Files already downloaded and verified
XorNeuronMLP_test
/home/yc/repo/xor_neuron_yoon/exp/xor_neuron_mlp_cifar_test/XorNeuronMLP_001_cifar10_2025-Dec-24-22-12-53
Loaded model from /home/yc/repo/xor_neuron_yoon/exp/xor_neuron_mlp_cifar_test/XorNeuronMLP_001_cifar10_2025-Dec-24-22-12-53/model_snapshot_best_phase2.pth to cuda


100%|██████████| 20/20 [00:00<00:00, 36.15it/s]
INFO  | 2025-12-24 22:17:16,322 | inference_runner.py       | line 1309 : Test Accuracy = 0.5056 +- 0
